In [12]:
import numpy as np
import pandas as pd
import re

In [13]:
from sklearn.model_selection import train_test_split
from sklearn.impute import KNNImputer
from sklearn.preprocessing import StandardScaler, OrdinalEncoder

In [14]:
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_squared_error, r2_score, mean_absolute_error, explained_variance_score, max_error, median_absolute_error

In [15]:
import mlflow
from mlflow.models.signature import infer_signature

mlflow.set_experiment("Random_Forest_Regressor_HPO")
mlflow.set_tracking_uri("http://localhost:5000")

In [16]:
import json
import optuna
optuna.logging.set_verbosity(optuna.logging.WARNING)
from optuna.samplers import TPESampler
from sklearn.experimental import enable_halving_search_cv
from sklearn.model_selection import RandomizedSearchCV, cross_val_score, HalvingRandomSearchCV

In [17]:
param_list = {}

In [18]:
# Define a function to calculate all metrics and format a report
def calculate_metrics(y, y_pred, y_val = None, y_pred_val = None, verbose=True):
    # Calculate train metrics
    mse = mean_squared_error(y, y_pred)
    r2 = r2_score(y, y_pred)
    mae = mean_absolute_error(y, y_pred)
    evs = explained_variance_score(y, y_pred)
    max_err = max_error(y, y_pred)
    medae = median_absolute_error(y, y_pred)
    metrics = {
        "mse": mse,
        "r2": r2,
        "mae": mae,
        "evs": evs,
        "max_err": max_err,
        "medae": medae
    }

    # If validation data is provided, calculate validation metrics
    if y_val is not None and y_pred_val is not None:
        # Calculate validation metrics
        mse_val = mean_squared_error(y_val, y_pred_val)
        r2_val = r2_score(y_val, y_pred_val)
        mae_val = mean_absolute_error(y_val, y_pred_val)
        evs_val = explained_variance_score(y_val, y_pred_val)
        max_err_val = max_error(y_val, y_pred_val)
        medae_val = median_absolute_error(y_val, y_pred_val)
        metrics.update({
            "mse_val": mse_val,
            "r2_val": r2_val,
            "mae_val": mae_val,
            "evs_val": evs_val,
            "max_err_val": max_err_val,
            "medae_val": medae_val
        })

        # Create a table for formatting metrics
        if verbose:
            print("Metric       |    Train     |  Validation")
            print("-------------|--------------|---------------")
            print(f"MSE          |   {mse:.2e}   |   {mse_val:.2e}")
            print(f"R²           |   {r2:.4f}     |   {r2_val:.4f}")
            print(f"MAE          |   {mae:.2e}   |   {mae_val:.2e}")
            print(f"EVS          |   {evs:.4f}     |   {evs_val:.4f}")
            print(f"Max Error    |   {max_err:.2e}   |   {max_err_val:.2e}")
            print(f"Median AE    |   {medae:.2e}   |   {medae_val:.2e}")
            print()
            
    elif verbose:
        print(f"MSE          |   {mse:.2e}   ")
        print(f"R²           |   {r2:.4f}     ")
        print(f"MAE          |   {mae:.2e}   ")
        print(f"EVS          |   {evs:.4f}     ")
        print(f"Max Error    |   {max_err:.2e}   ")
        print(f"Median AE    |   {medae:.2e}   ")
        print()
    
    # Return metrics in a dictionary for reporting
    return metrics

## Preparacion de datos

In [19]:
df = pd.read_csv('data/train/cleaned_train.csv')

# Convert the 'Moneda' column to binary (1 for 'US$', 0 for '$')
df['Moneda'] = df['Moneda'].apply(lambda x: 1 if x == 'US$' else 0)
df.loc[df['Moneda'] == 1, 'Precio'] = df.loc[df['Moneda'] == 1, 'Precio'] * 1100

# Convert Kilometers column to numeric, removing 'km' if present
df['Kilómetros'] = df['Kilómetros'].str.replace(' km', '', regex=False).str.replace('km', '', regex=False)
df['Kilómetros'] = pd.to_numeric(df['Kilómetros'], errors='coerce')

# Handle the price conversion for US$ values (Moneda = 1)
# Assuming 1 US$ = 1100 local currency

def extraer_formato_correcto(texto):
    if isinstance(texto, str):
        match = re.search(r'\b(\d\.\d)', texto, re.IGNORECASE)
        return match.group(1) if match else np.nan
    return np.nan

# Aplicar solo sobre la columna 'Motor'
df['Motor'] = df['Motor'].apply(extraer_formato_correcto).astype(float)

df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 14603 entries, 0 to 14602
Data columns (total 21 columns):
 #   Column                   Non-Null Count  Dtype  
---  ------                   --------------  -----  
 0   idx                      14603 non-null  int64  
 1   Marca                    14603 non-null  object 
 2   Modelo                   14603 non-null  object 
 3   Año                      14603 non-null  float64
 4   Versión                  14085 non-null  object 
 5   Color                    14288 non-null  object 
 6   Tipo de combustible      14603 non-null  object 
 7   Puertas                  14603 non-null  float64
 8   Transmisión              14590 non-null  object 
 9   Motor                    14410 non-null  float64
 10  Tipo de carrocería       14603 non-null  object 
 11  Con cámara de retroceso  3749 non-null   object 
 12  Kilómetros               14603 non-null  float64
 13  Título                   14603 non-null  object 
 14  Precio                

In [20]:
# Variable target
y = df["Precio"]
X = df.drop(columns=["idx", "Tipo de carrocería", "Título", "Descripción", "Precio"])

# Train/val split
X_train, X_val, y_train, y_val = train_test_split(X, y, test_size=0.2, random_state=42)

# Target/mean encoding (usar media de train solamente)
mean_encoded_cols = ["Marca", "Modelo", "Versión", "Color"]
for col in mean_encoded_cols:
    means = X_train.join(y_train).groupby(col)["Precio"].mean()
    X_train[col] = X_train[col].map(means)
    X_val[col] = X_val[col].map(means)  # aplicar los mismos valores (NaN si valor nuevo)

# Categóricas para one-hot
one_hot_cols = ["Tipo de combustible", "Transmisión", "Con cámara de retroceso", "Tracción", "Tipo de vendedor"]

# Normalizamos las columnas numéricas
num_cols = X_train.select_dtypes(include=[np.number]).columns.tolist()
scaler = StandardScaler()
X_train[num_cols] = scaler.fit_transform(X_train[num_cols])
X_val[num_cols] = scaler.transform(X_val[num_cols])  # usar la misma escala

# Ordinal encode antes de KNN imputación
encoder = OrdinalEncoder(handle_unknown='use_encoded_value', unknown_value=-1)
X_train[one_hot_cols] = encoder.fit_transform(X_train[one_hot_cols])
X_val[one_hot_cols] = encoder.transform(X_val[one_hot_cols])  # usar misma codificación

# Imputación KNN solo con train
imputer = KNNImputer(n_neighbors=5)
X_train_imputed = imputer.fit_transform(X_train)
X_val_imputed = imputer.transform(X_val)

# Volvemos a DataFrame
X_train = pd.DataFrame(X_train_imputed, columns=X.columns, index=X_train.index)
X_val = pd.DataFrame(X_val_imputed, columns=X.columns, index=X_val.index)

# One-hot encoding (sin dummy_na)
X_train = pd.get_dummies(X_train, columns=one_hot_cols, dtype=np.float64)
X_val = pd.get_dummies(X_val, columns=one_hot_cols, dtype=np.float64)

# Alineamos columnas (puede faltar alguna en val que está en train)
X_val = X_val.reindex(columns=X_train.columns, fill_value=0)

print("Train:", X_train.shape)
print("Val:", X_val.shape)

Train: (11682, 39)
Val: (2921, 39)


## First Experiment

Ajuste de hiperparametros

In [21]:
# Espacio de hiperparámetros para Random Forest
param_dist = {
    'n_estimators': [100, 200, 300, 400],
    'max_depth': [3, 4, 5, 6, 7, 8, None],
    'min_samples_split': [2, 5, 10],
    'min_samples_leaf': [1, 2, 4],
    'max_features': ['sqrt', 'log2', None],
    'bootstrap': [True],
    'max_samples': [0.5, 0.7, 0.9, None]
}

# Modelo base
rf = RandomForestRegressor(random_state=42)

# RandomizedSearchCV
search = RandomizedSearchCV(
    estimator=rf,
    param_distributions=param_dist,
    n_iter=50,
    scoring='neg_root_mean_squared_error',
    cv=3,
    verbose=1,
    random_state=42,
    n_jobs=-1
)

# Entrenar
search.fit(X_train, y_train)
best_model = search.best_estimator_
best_params = search.best_params_

# Predicciones
y_pred_train = best_model.predict(X_train)
y_pred_val = best_model.predict(X_val)

calculate_metrics(y_train, y_pred_train, y_val, y_pred_val)

param_list["RandomizedSearchCV"] = best_params

print("Best parameters found:")
print(json.dumps(best_params, indent=4))


Fitting 3 folds for each of 50 candidates, totalling 150 fits


Metric       |    Train     |  Validation
-------------|--------------|---------------
MSE          |   1.43e+13   |   5.65e+13
R²           |   0.9716     |   0.8750
MAE          |   1.27e+06   |   2.90e+06
EVS          |   0.9716     |   0.8750
Max Error    |   2.05e+08   |   2.76e+08
Median AE    |   6.07e+05   |   1.38e+06

Best parameters found:
{
    "n_estimators": 200,
    "min_samples_split": 2,
    "min_samples_leaf": 1,
    "max_samples": 0.9,
    "max_features": null,
    "max_depth": null,
    "bootstrap": true
}


In [ ]:
# Espacio de hiperparámetros refinado para Random Forest
param_dist = {
    'n_estimators': [250, 300, 350, 400],
    'max_depth': [5, 6, 7, 8, 9],
    'min_samples_split': [2, 3, 4, 5],
    'min_samples_leaf': [1, 2, 3],
    'max_features': ['sqrt', 'log2', 0.6, 0.7, 0.8],
    'bootstrap': [True],
    'max_samples': [0.7, 0.8, 0.9, None],
    'criterion': ['squared_error', 'absolute_error', 'poisson']
}

# Modelo base
rf = RandomForestRegressor(random_state=42)

# RandomizedSearchCV
search = RandomizedSearchCV(
    estimator=rf,
    param_distributions=param_dist,
    n_iter=50,
    scoring='neg_root_mean_squared_error',
    cv=3,
    verbose=1,
    random_state=42,
    n_jobs=-1
)

# Entrenar
search.fit(X_train, y_train)
best_model = search.best_estimator_
best_params = search.best_params_

# Predicciones
y_pred_train = best_model.predict(X_train)
y_pred_val = best_model.predict(X_val)

# Crear reporte
calculate_metrics(y_train, y_pred_train, y_val, y_pred_val)
param_list["RandomizedSearchCV_2"] = best_params

print("Best parameters found:")
print(json.dumps(best_params, indent=4))

Fitting 3 folds for each of 50 candidates, totalling 150 fits


In [ ]:
def objective(trial):
    params = {
        'max_depth': trial.suggest_int('max_depth', 3, 15),
        'n_estimators': trial.suggest_int('n_estimators', 100, 500),
        'min_samples_split': trial.suggest_int('min_samples_split', 2, 10),
        'min_samples_leaf': trial.suggest_int('min_samples_leaf', 1, 10),
        'max_features': trial.suggest_categorical('max_features', ['sqrt', 'log2', None]),
        'bootstrap': trial.suggest_categorical('bootstrap', [True]),
        'criterion': trial.suggest_categorical('criterion', ['squared_error', 'absolute_error', 'poisson']),
    }
    
    # Only add max_samples if bootstrap is True
    if params['bootstrap']:
        params['max_samples'] = trial.suggest_float('max_samples', 0.5, 1.0)
    
    model = RandomForestRegressor(**params, random_state=42)
    score = cross_val_score(model, X_train, y_train, cv=3, scoring='neg_root_mean_squared_error').mean()
    return score

optuna.logging.set_verbosity(optuna.logging.WARNING)

study = optuna.create_study(direction="minimize", sampler=TPESampler())
study.optimize(objective, n_trials=50)
best_params = study.best_params

# entrenar el modelo con los mejores hiperparámetros
best_model = RandomForestRegressor(**best_params, random_state=42)
best_model.fit(X_train, y_train)
# Predicciones
y_pred_train = best_model.predict(X_train)
y_pred_val = best_model.predict(X_val)
# Crear reporte
calculate_metrics(y_train, y_pred_train, y_val, y_pred_val)
param_list["Optuna"] = best_params
print("Best parameters found:")
print(json.dumps(best_params, indent=4))

In [ ]:
# Espacio de hiperparámetros para Random Forest
param_dist = {
    'n_estimators': [100, 200, 300, 400],
    'max_depth': [3, 5, 7, 9, None],
    'min_samples_split': [2, 5, 10],
    'min_samples_leaf': [1, 2, 4],
    'max_features': ['sqrt', 'log2', None],
    'bootstrap': [True],
    'criterion': ['squared_error', 'absolute_error', 'poisson']
}

# Modelo base
rf = RandomForestRegressor(random_state=42)

halving_search = HalvingRandomSearchCV(
    estimator=rf,
    param_distributions=param_dist,
    factor=3,
    cv=3,
    scoring='neg_root_mean_squared_error',
    random_state=42,
    n_jobs=-1
)
halving_search.fit(X_train, y_train)
best_model = halving_search.best_estimator_
best_params = halving_search.best_params_

# Predicciones
y_pred_train = best_model.predict(X_train)
y_pred_val = best_model.predict(X_val)
# Crear reporte
calculate_metrics(y_train, y_pred_train, y_val, y_pred_val)

param_list["HalvingRandomSearchCV"] = best_params

print("Best parameters found:")
print(json.dumps(best_params, indent=4))


Metric       |    Train     |  Validation
-------------|--------------|---------------
MSE          |   1.85e+13   |   4.68e+13
R²           |   0.9634     |   0.8966
MAE          |   2.24e+06   |   2.90e+06
EVS          |   0.9634     |   0.8966
Max Error    |   2.19e+08   |   2.37e+08
Median AE    |   1.45e+06   |   1.60e+06



In [ ]:
input_example = X_train.iloc[0:1]
artifact_csv_path = "data/train/cleaned_train.csv"  # ajusta la ruta si es necesario

for name, params in param_list.items():
    model = RandomForestRegressor(random_state=42, **params)
    model.fit(X_train, y_train)

    y_pred_train = model.predict(X_train)
    y_pred_val = model.predict(X_val)

    metrics = calculate_metrics(y_val, y_pred_val, verbose=False)

    run_name = f"RandomForest_{name}"

    with mlflow.start_run(run_name=run_name):
        mlflow.log_params(params)
        mlflow.log_metrics(metrics)
        signature = infer_signature(input_example, model.predict(input_example))
        mlflow.sklearn.log_model(model, artifact_path="model", input_example=input_example, signature=signature)

2025/06/15 23:54:16 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.


MSE          |   3.92e+13   
R²           |   0.9133     
MAE          |   2.73e+06   
EVS          |   0.9134     
Max Error    |   2.01e+08   
Median AE    |   1.45e+06   



[23:54:17] WARNING: /workspace/src/c_api/c_api.cc:1427: Saving model in the UBJSON format as default.  You can use file extension: `json`, `ubj` or `deprecated` to choose between formats.


🏃 View run XGBoost_RandomizedSearchCV at: http://localhost:5000/#/experiments/289899895095468934/runs/80e0b370061445129253578b564ec56a
🧪 View experiment at: http://localhost:5000/#/experiments/289899895095468934


2025/06/15 23:54:21 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.


...

🏃 View run XGBoost_RandomizedSearchCV_2 at: http://localhost:5000/#/experiments/289899895095468934/runs/4aa51d2c9b5a4a53ae20548ab2a1c4ac
🧪 View experiment at: http://localhost:5000/#/experiments/289899895095468934


2025/06/15 23:54:24 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.


MSE          |   4.68e+13   
R²           |   0.8966     
MAE          |   2.90e+06   
EVS          |   0.8966     
Max Error    |   2.37e+08   
Median AE    |   1.60e+06   



[23:54:24] WARNING: /workspace/src/c_api/c_api.cc:1427: Saving model in the UBJSON format as default.  You can use file extension: `json`, `ubj` or `deprecated` to choose between formats.


🏃 View run XGBoost_HalvingRandomSearchCV at: http://localhost:5000/#/experiments/289899895095468934/runs/631a2c1c66d74c909756958c94983c37
🧪 View experiment at: http://localhost:5000/#/experiments/289899895095468934
